In [1]:
import sys
sys.path.insert(0, "../src")
import multiprocessing
from itertools import repeat
import pickle
import os

import random
import numpy as np
import scipy
from scipy import interpolate
import pandas as pd
from datetime import datetime, timedelta

from skorch import NeuralNetRegressor
from skorch import NeuralNet
from skorch import callbacks
from skorch import dataset as skorch_dataset
from skorch.callbacks import Checkpoint
import torch
from torch import nn
from sklearn.preprocessing import MinMaxScaler # data preparation

# Asymmetric Laplace imports:
import asymmLaplace_accrue_torch
import NN_regression_AL_3D
from scipy.stats import laplace_asymmetric

# Two-piece Gaussian imports:
import twoPieceGauss_accrue_torch
import NN_regression_TPG_3D
from twopiece.scale import *
from twopiece.shape import *
from twopiece.double import *

In [ ]:
forecast_date_start = "2022-01-01"
forecast_date_mid1 = "2022-07-01"
forecast_date_mid2 = "2023-02-01"
forecast_date_end = "2023-07-01"
# forecast_date_end = "2022-01-02"
hours = pd.date_range(f"{forecast_date_start} 00:00", f"{forecast_date_end} 00:00", freq="1H")

In [ ]:
def get_HRRR_data():
    df1 = pd.read_csv("../DIA_data/DIA_HRRR_f01_vs_Obs"+str(forecast_date_start)+"_"+str(forecast_date_mid1)+".csv")
    df2 = pd.read_csv("../DIA_data/DIA_HRRR_f01_vs_Obs"+str(forecast_date_mid1)+"_"+str(forecast_date_mid2)+".csv")
    df3 = pd.read_csv("../DIA_data/DIA_HRRR_f01_vs_Obs"+str(forecast_date_mid2)+"_"+str(forecast_date_end)+".csv")
    df = pd.concat([df1, df2, df3], ignore_index=True)
    df.rename(columns={"Unnamed: 0": "timestamp"}, inplace=True)
    num_hours, num_var = np.shape(df)
    dim_test = 180*24+1
    dim_train = num_hours - dim_test
    return df.head(dim_train), df.tail(dim_test)

In [4]:
def beta_calibration(b, x_train, y_train, dist, out_dir):
    """
    Given a beta-value and training data, learn var1 and var2 for the
    distribution.
    input:
        b: beta-value for ACCRUE loss function (b \in [0,1])
        x_train: input to NN
        y_train: pairs of obs, pred
        dist: error distribution the NN is assuming (current options AL or TPG)
    output:
        [crps, rs]: the CRPS and RS values for the training data of the
                    optimized NN
    """
    my_params = out_dir+"params"+str(b)+".pt"

    net_regr=None
    if dist=="AL":
        net_regr = NN_regression_AL_3D.VarNet(
        module=NN_regression_AL_3D.RegressorModule,
        beta=b,
        max_epochs=1000,
        optimizer=torch.optim.Adam,
        # try a different opt
        # try different batch size
        batch_size=500, 
        lr=1e-3, # could change learning rate to variable or decrease
        callbacks=[callbacks.EarlyStopping(monitor='valid_loss', patience=10),
                    Checkpoint(f_params=my_params)],
        train_split=skorch_dataset.ValidSplit(0.2),
        verbose=0
        )
    elif dist=="TPG":
        net_regr = NN_regression_TPG_3D.VarNet(
        module=NN_regression_TPG_3D.RegressorModule,
        beta=b,
        # kappa=true_kappa,
        max_epochs=1000,
        optimizer=torch.optim.Adam,
        lr=1e-3,
        batch_size=500,
        callbacks=[callbacks.EarlyStopping(monitor='valid_loss', patience=10),
                    Checkpoint(f_params=my_params)],
        train_split=skorch_dataset.ValidSplit(0.2), # 2 would be train set is first 1/2 and valid second 1/2
        verbose=0
        )
    else:
        raise NameError("Undefined distribution!")

    net_regr.initialize_criterion()
    net_regr.criterion_
    
    net_regr.fit(x_train, y_train)
    net_regr.load_params(f_params=my_params)

    pred_all = net_regr.predict(x_train)
    pred_all = np.exp(pred_all)
    # print(pred_all)
    pred_var1 = pred_all[:,0]
    pred_var2 = pred_all[:,1]
    # pred_lam=pred_all

    eps = y_train[:,0] - y_train[:,1]
    crps=-1
    rs=-1
    if dist=="AL":
        crps = asymmLaplace_accrue_torch.get_avg_CRPS_torch(
                            torch.tensor(pred_var1),
                            torch.tensor(eps), lam=torch.tensor(pred_var2))
        rs = asymmLaplace_accrue_torch.analytical_RS_torch(
                            torch.tensor(eps),
                            torch.tensor(pred_var1), torch.tensor(pred_var2))
    elif dist=="TPG":
        crps = twoPieceGauss_accrue_torch.get_avg_CRPS_torch(
                            torch.tensor(pred_var1),
                            torch.tensor(pred_var2), torch.tensor(eps))
        rs = twoPieceGauss_accrue_torch.analytical_RS_torch(
                            torch.tensor(eps),
                            torch.tensor(pred_var1), torch.tensor(pred_var2))
    print("b:", b, crps, rs)
    return [crps, rs]

In [5]:
def get_optimal_beta(dist,x,y,f,out_dir):
    """
    Given calibration data, find the best beta-value (minimizes ACCRUE)
    input:
        dist: error distribution (AL or TPG)
        x, y, f: training input, obs, and predictions
        out_file: file to write resulting info into
    output:
        beta: optimal beta value for the calibration data
    """
    out_file = out_dir+"opt_beta.txt"

    # # loop through beta values to select the "best" given calibration data
    betas = np.arange(0.1, 1.0, 0.1)
    # betas = np.arange(0.25, 1.0, 0.25)
    crps_beta = np.zeros(len(betas))
    rs_beta = np.zeros(len(betas))
    y_train = np.vstack([y.T, f.T]).T
    y_train = y_train.astype(np.float32)
    # x_train = np.reshape(x, (-1,1))
    # x_train = x_train.astype(np.float32)
    x_train = x.astype(np.float32)

    # with multiprocessing.Pool(len(betas)) as p:
    #     result= np.array(p.starmap(beta_calibration, zip(betas, repeat(x_train),
    #                                 repeat(y_train), repeat(dist), repeat(out_dir))))
    #     crps_beta=result[:,0]
    #     rs_beta=result[:,1]

    for i in range(len(betas)):
        result = beta_calibration(betas[i], x_train, y_train, dist, out_dir)
        crps_beta[i]=result[0]
        rs_beta[i]=result[1]

    a = rs_beta**2
    b = crps_beta**2
    c = np.sqrt(a+b)
    # ind is the index in betas ordered by c, so if betas[7] is optimal
    # ind[0]=7
    ind = np.argsort(c)

    info = np.array([ind, betas, crps_beta, rs_beta])
    np.savetxt(out_file, info)
    return betas[ind[0]]

In [6]:
def get_best_net(dist,x_train, y_train, x_test, y_test, beta, out_dir):
    """
    Generate 2 random NN, and return the one that minimizes the testing data's
    loss
    """
    best_net = None
    min_loss = np.Inf
    i = 0
    # gaurantee at least five starting config works!
    # RB: this loop could also be parallelized
    # while (i < 5): #or (best_net == None and i >= 5):
    while (i < 2) or (best_net == None and i >= 2):
    # while (i < 1):
        print("replicate:", i)
        my_params=out_dir+"params_"+str(i)+".pt"
        net_i=None
        if dist=="AL":
            net_i = NN_regression_AL_3D.VarNet(
                module=NN_regression_AL_3D.RegressorModule,
                beta=beta,
                max_epochs=5000,
                optimizer=torch.optim.Adam,
                lr=1e-3,
                batch_size=32,
                callbacks=[callbacks.EarlyStopping(monitor='valid_loss',
                           patience=10),
                           Checkpoint(f_params=my_params)],
                # need to save the best model not the last
                train_split=skorch_dataset.ValidSplit(0.2),
                verbose=0
            )
        elif dist=="TPG":
            net_i = NN_regression_TPG_3D.VarNet(
                module=NN_regression_TPG_3D.RegressorModule,
                beta=beta,
                max_epochs=5000,
                # optimizer=torch.optim.RMSprop,
                # optimizer=torch.optim.SGD,
                optimizer=torch.optim.Adam,
                lr=1e-3,
                batch_size=32,
                callbacks=[callbacks.EarlyStopping(monitor='valid_loss',
                           patience=10),
                           Checkpoint(f_params=my_params)],
                train_split=skorch_dataset.ValidSplit(0.2),
                verbose=0
            )


        # make sure the loss function is ACCRUE!
        net_i.initialize_criterion()
        net_i.criterion_

        print("x", x_train.shape)  # should be (3623, 3)
        print("y",y_train.shape)  # should be (3623,)

        net_i.fit(x_train, y_train)
        net_i.load_params(f_params=my_params)
        sd_i = net_i.predict(x_test)
        ar_i = net_i.get_loss(torch.from_numpy(sd_i), torch.from_numpy(y_test),
                                X=torch.from_numpy(x_test))
        ar_i = ar_i.detach().numpy()
        print(ar_i)
        if not(np.isnan(ar_i)) and ar_i < min_loss:
            best_net = net_i
            min_loss = ar_i

        if ar_i != np.nan:
            i += 1

    return best_net, min_loss

In [7]:
def learn_dist(i, dist, beta, out_dir, x_interp,
               x_train, y_train, hres_train, x_test, y_test, hres_test):
    """
    learn parameters var1 and var2 of dist given the data.
    Output:
        [var1(x_test), var2(x_test)]: the parameter outputs interpolated to
                                          the given inputs
    """
    np.random.seed(seed=i)
    random.seed(i)
    torch.manual_seed(i)

    # RB: adding transformation of inputs of the NN
    # instantiate scaler objects for inputs/outputs
    # input_scaler = MinMaxScaler()
    # # output_scaler = MinMaxScaler()
    # # MinMaxScaler object takes parameter: array-like of shape (n_samples, n_features)
    # x_train = input_scaler.fit_transform(np.reshape(x_train, (-1, 1))).flatten()
    # x_test = input_scaler.transform(np.reshape(x_test, (-1, 1))).flatten()
    # y_train = output_scaler.fit_transform(np.reshape(y_train, (-1, 1))).flatten()
    # y_test = output_scaler.transform(np.reshape(y_test, (-1, 1))).flatten()

    # eps = get_error(y_train, hres_train)

    # reformatting for PyTorch
    y_train = np.vstack([y_train.T, hres_train.T]).T
    y_train = y_train.astype(np.float32)
    # x_train = np.reshape(x_train, (-1,1))
    x_train = x_train.astype(np.float32)
    y_test = np.vstack([y_test.T, hres_test.T]).T
    y_test = y_test.astype(np.float32)
    # x_test = np.reshape(x_test, (-1,1))
    x_test = x_test.astype(np.float32)
    
    net, test_loss = get_best_net(dist, x_train, y_train, x_test, y_test, beta,
                                  out_dir)

    print("testing loss:", test_loss)

    with open(out_dir+"nets.pkl", 'ab') as f:
        pickle.dump(net, f)
    
    return

    # # calc parameters on all data
    # x_interp = input_scaler.transform(np.reshape(x_interp, (-1, 1))).flatten()
    # pred_all = np.exp(net.predict(x_interp))
    # # pred_all = net.predict(x_test)
    # pred_var1 = pred_all[:,0]
    # pred_var2 = pred_all[:,1]
    
    # return [pred_var1, pred_var2]

In [8]:
dist="TPG"
out_dir = "../output_HRRR_dewpt"+"_"+dist+"/"
train_data, test_data = get_HRRR_data()

# using observed dewpoint from 1hr ago as the input variable to ACCRUE
# x_train = train_data["obs_dewpt_C"].to_numpy()
# x_test = test_data["obs_dewpt_C"].to_numpy()
x_train = train_data[["obs_dewpt_C", "obs_wind_speed_mps", "obs_station_pressure_hPa"]].to_numpy()
x_test = test_data[["obs_dewpt_C", "obs_wind_speed_mps", "obs_station_pressure_hPa"]].to_numpy()
x_test = np.concatenate(([x_train[-1]], x_test[0:-1]))
x_train = x_train[0:-1]

y_train = train_data["obs_temp_C"].to_numpy()
y_train = y_train[1:]
y_test = test_data["obs_temp_C"].to_numpy()

hrrr_train = train_data["forecast_temp_C"].to_numpy()
hrrr_train = hrrr_train[1:]
hrrr_test = test_data["forecast_temp_C"].to_numpy()

x_interp = np.concatenate((x_train, x_test))

i = 1 # random seed

In [9]:
beta = get_optimal_beta(dist,x_train,y_train,hrrr_train,out_dir)

b: 0.1 tensor(0.8104) tensor(0.6676)
b: 0.2 tensor(0.8049) tensor(0.6682)
b: 0.30000000000000004 tensor(0.8038) tensor(0.6689)
b: 0.4 tensor(0.8083) tensor(0.6691)
b: 0.5 tensor(0.8000) tensor(0.6696)
b: 0.6 tensor(0.8056) tensor(0.6697)
b: 0.7000000000000001 tensor(0.8007) tensor(0.6704)
b: 0.8 tensor(0.7983) tensor(0.6703)
b: 0.9 tensor(0.8060) tensor(0.6702)


In [10]:
beta

0.8

In [11]:
# var1,var2 = learn_dist(i, dist, beta, out_dir, x_interp,
#                         x_train, y_train, hres_train,
#                         x_test, y_test, hres_test)
learn_dist(i, dist, beta, out_dir, x_interp,
                        x_train, y_train, hrrr_train,
                        x_test, y_test, hrrr_test)


replicate: 0
x (8783, 3)
y (8783, 2)
0.9663566
replicate: 1
x (8783, 3)
y (8783, 2)
0.9665012
testing loss: 0.9663566
